In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), '..', '00-setup'))
from spark_session import get_spark, output_path, stop_and_wait
from seed_data import load_all, register_views
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = get_spark("23-duplicate-deep-dive")
dfs = register_views(spark)
emp = dfs["employees"]
sal = dfs["salary_history"]
pa  = dfs["project_assignments"]
po  = dfs["purchase_orders"]
pr  = dfs["performance_reviews"]
ee  = dfs["emp_events"]
lr  = dfs["leave_requests"]

# ── Section 1 ── Exact duplicates in salary_history (including surrogate key)
# Flaw: rows 15,16 are exact duplicates for emp 5, 2022-04-01 — only hist_id differs
# distinct() on ALL columns won't catch this because the surrogate key differs
print("sal.count():", sal.count())               # 33
print("sal.distinct().count():", sal.distinct().count())  # still 33 (hist_id differs!)
# Must check business key only
biz_cols = ["emp_id", "salary_before", "salary_after", "effective_date", "change_reason", "changed_by"]
sal.groupBy(biz_cols) \
   .agg(F.count("*").alias("cnt"), F.collect_list("hist_id").alias("hist_ids")) \
   .filter(F.col("cnt") > 1).show(truncate=False)


# ── Section 2 ── Exact duplicates in project_assignments
# Flaw: rows 25,26 are exact duplicates (same emp, project, role, dates, hours)
pa_biz = ["emp_id", "project_id", "role", "start_date", "end_date", "hours_billed"]
pa.groupBy(pa_biz) \
  .agg(F.count("*").alias("cnt"), F.collect_list("assignment_id").alias("ids")) \
  .filter(F.col("cnt") > 1).show(truncate=False)


# ── Section 3 ── Exact duplicates in purchase_orders
# Flaw: rows 19,20 are exact duplicates (same dept, vendor, category, amount, date)
po_biz = ["dept_id", "vendor", "item_category", "amount", "order_date"]
po.groupBy(po_biz) \
  .agg(F.count("*").alias("cnt"), F.collect_list("order_id").alias("ids")) \
  .filter(F.col("cnt") > 1).show(truncate=False)


# ── Section 4 ── Soft duplicates in employees (same entity, different emp_id)
# Flaw: emp 18 and emp 19 share last_name, dept_id, hire_date, job_title — likely the same person
# emp 19 also has salary=0.0, suggesting it was created in error
soft_keys = ["last_name", "dept_id", "hire_date", "job_title"]
emp.groupBy(soft_keys) \
   .agg(
       F.count("*").alias("cnt"),
       F.collect_list("emp_id").alias("emp_ids"),
       F.collect_list("salary").alias("salaries"),
   ).filter(F.col("cnt") > 1).show(truncate=False)


# ── Section 5 ── Dedup impact on window functions (running sum with dup)
# Flaw: emp 5's duplicate row on 2022-04-01 inflates running totals
# Shows WHY dedup must happen before window functions, not after
w_run = Window.partitionBy("emp_id").orderBy("effective_date", "hist_id")
sal.withColumn("running_total", F.sum("salary_after").over(w_run)) \
   .filter(F.col("emp_id") == 5) \
   .select("hist_id", "effective_date", "salary_after", "running_total").show()
# The duplicate row causes salary_after to be double-counted in the running total


# ── Section 6 ── Dedup via ROW_NUMBER (canonical approach)
# ROW_NUMBER() over business key, ordered by surrogate key → keep row 1 → deterministic
w_dup = Window.partitionBy(*biz_cols).orderBy("hist_id")
sal_clean = sal.withColumn("rn", F.row_number().over(w_dup)) \
               .filter(F.col("rn") == 1).drop("rn")
print(f"After dedup: {sal.count()} → {sal_clean.count()}")
# Re-run window function on clean data — running total is now correct
sal_clean.withColumn("running_total", F.sum("salary_after").over(w_run)) \
         .filter(F.col("emp_id") == 5) \
         .select("hist_id", "effective_date", "salary_after", "running_total").show()


# ── Section 7 ── Duplicates in sessionization context (emp_events)
# Flaw: emp 14 has 3 events with session_id=None — cannot rely on session_id column
# Pattern: derive session boundaries from time gaps instead of the session_id field
w_sess = Window.partitionBy("emp_id").orderBy("event_ts")
ee_sess = ee.withColumn("prev_ts", F.lag("event_ts", 1).over(w_sess)) \
            .withColumn("gap_min",
                (F.col("event_ts").cast("long") - F.col("prev_ts").cast("long")) / 60) \
            .withColumn("new_session",
                F.when(F.col("gap_min").isNull() | (F.col("gap_min") > 30), 1).otherwise(0)) \
            .withColumn("session_num",
                F.sum("new_session").over(w_sess.rowsBetween(Window.unboundedPreceding, Window.currentRow)))
ee_sess.filter(F.col("emp_id") == 14) \
       .select("emp_id", "event_type", "event_ts", "session_num").show()


# ── Section 8 ── Overlapping date ranges in leave_requests (logical duplicate)
# Flaw: emp 12 has requests 1 and 2 whose date ranges overlap — a booking conflict
# Pattern: self-join on same emp_id where ranges intersect (start_a <= end_b AND end_a >= start_b)
lr.alias("a").join(lr.alias("b"),
    (F.col("a.emp_id")      == F.col("b.emp_id")) &
    (F.col("a.request_id")   < F.col("b.request_id")) &
    (F.col("a.start_date")  <= F.col("b.end_date")) &
    (F.col("a.end_date")    >= F.col("b.start_date"))
).select(
    F.col("a.request_id").alias("req_1"),
    F.col("a.start_date").alias("start_1"),
    F.col("a.end_date").alias("end_1"),
    F.col("b.request_id").alias("req_2"),
    F.col("b.start_date").alias("start_2"),
    F.col("b.end_date").alias("end_2"),
    F.col("a.emp_id"),
).show()
# Expected: emp 12 requests 1 and 2 overlap


# ── Section 9 ── Dedup strategy summary
# Strategy 1: dropDuplicates(subset=biz_cols)
#   — fastest, keeps arbitrary row within group; non-deterministic which surrogate survives
# Strategy 2: ROW_NUMBER() partitioned by biz_cols, ordered by surrogate key
#   — deterministic; keep min/max surrogate; best when original row structure is needed
# Strategy 3: null_score ranking — add a score = count of non-null cols, keep highest score
#   — use when some dup rows are more complete than others and you want the richest record
# Strategy 4: groupBy(biz_cols) + agg(...)
#   — when you need a summarized result rather than the individual rows
# Rule of thumb:
#   need original row? → ROW_NUMBER
#   need aggregates?   → groupBy + agg
#   most complete row? → null_score + ROW_NUMBER


stop_and_wait(spark)